# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: _TODO — exact name as it appears in TP-LMS_
* Student ID: 2503051A


**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

**Use of Generative AI**

Claude Code was used for project scaffolding, code review and structuring. All modelling decisions, metric justification and analysis are my own and I can explain each of them. _(Declaring this does not affect marks — see spec, Guidelines on using Gen AI.)_


# Libraries

In [ ]:
## Import libraries
import json
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff
from scipy.stats import spearmanr

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import (
    train_test_split, cross_validate, StratifiedKFold, RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, make_scorer,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)

# Fixed seed everywhere: the rubric's tuning row requires a reproducible setup.
RANDOM_STATE = 42

ARTIFACTS = Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


# 1. Business Understanding

**Goal:** flag phishing websites from 30 structural signals extracted from a URL and its page, so a browser extension / email gateway can warn a user *before* they enter credentials.

**Task type:** binary classification. Target `Result` — `-1` phishing, `1` legitimate.

**Cost of the two errors (this drives the metric choice in section 5):**

| Error | What happens | Who pays |
| --- | --- | --- |
| False negative — phishing scored legitimate | user is not warned, enters credentials | account takeover, financial loss |
| False positive — legitimate scored phishing | user sees a warning on a safe site | one dismissed banner |

These costs are **not symmetric**, and that asymmetry — not the class ratio — is why accuracy is the wrong headline metric here. The classes are close to balanced (confirmed in 2.3.1.1), so accuracy is not *broken*; it is simply blind to which kind of mistake was made.

> **TODO (my own words):** expand the paragraph above into your own framing of who the customer is and what a wrong prediction costs them. This is the paragraph section 5 points back to, and the one the investor Q&A will push on.

**Assumptions**

> **TODO (my own words):** state them, e.g. the 30 features can be recomputed cheaply at request time; phishing tactics in the training data still resemble today's; a warning is advisory, not a hard block.


# 2. Data Understanding

## 2.1 Load dataset

Source: UCI Phishing Websites, id=327 (https://archive.ics.uci.edu/dataset/327/phishing+websites). Tutor-approved at Checkpoint 1. 11,055 rows x 30 features + target.

Loaded from `Training Dataset.arff`. **Not** `.old.arff` — that is a different, 2,456-row file from the same download that encodes binary features as {0,1} instead of {-1,1}, so it would silently change what every feature means.


In [ ]:
## Read the ARFF file into a pandas DataFrame
raw, meta = arff.loadarff('Training Dataset.arff')
df = pd.DataFrame(raw)

# scipy returns ARFF nominal values as bytes (b'-1'); convert to signed ints.
df = df.apply(lambda s: s.str.decode('utf-8')).astype(int)

TARGET = 'Result'
FEATURES = [c for c in df.columns if c != TARGET]

print(f'shape: {df.shape}  |  {len(FEATURES)} features')
df.head()


## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
df.info()


In [ ]:
## Check for missing data
print('missing values:', df.isna().sum().sum())
print('duplicate rows:', df.duplicated().sum())


In [ ]:
## Describe data distribution
# Every column is a nominal code, so value counts are more informative than
# mean/std — a 'mean' of -0.3 on a {-1,1} column has no meaning.
cardinality = df[FEATURES].nunique().value_counts().rename('n_features')
print(cardinality.rename_axis('n_distinct_values'))

df.describe().T


> **TODO (my own words):** note what the cardinality split tells you: 22 features are binary {-1,1} and 8 are ternary {-1,0,1}. Say why this rules out histograms/scaling and points to crosstabs instead.


## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
counts = df[TARGET].value_counts().sort_index()
labels = {-1: 'Phishing', 1: 'Legitimate'}

ax = sns.barplot(x=[labels[i] for i in counts.index], y=counts.values)
ax.set(title='Class balance of Result', xlabel='', ylabel='Websites')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,} ({v / len(df):.1%})', ha='center', va='bottom')
plt.show()

ratio = counts.max() / counts.min()
print(f'majority:minority = {ratio:.2f}:1')


> **TODO (my own words):** interpret. The ratio is ~1.26:1 — near balanced. Be explicit that this means you are **not** choosing recall because of imbalance; you are choosing it because of the cost asymmetry in section 1. Also note what a majority-class baseline would therefore score (~56%), which is the number section 4 has to beat.


### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features
# Every feature is a nominal code, so "distribution" on its own says little — a bar
# chart of a {-1,1} column just shows two bars. The question that carries information
# is conditional: *within each level of a feature, what share of sites are phishing?*
# The overall phishing rate is the reference; a level sitting far from it carries
# signal, a level sitting on it carries none.

BASE_RATE = (df[TARGET] == -1).mean()
print(f'overall phishing rate: {BASE_RATE:.1%}')


def phishing_rate_table(frame, features, target=TARGET):
    """One row per (feature, level): how many sites sit at that level, and what
    share of them are phishing."""
    rows = []
    for f in features:
        ct = pd.crosstab(frame[f], frame[target])
        for level, counts in ct.iterrows():
            n = counts.sum()
            rows.append({'feature': f, 'level': level, 'n': n,
                         'share_of_rows': n / len(frame),
                         'phishing_rate': counts.get(-1, 0) / n})
    return pd.DataFrame(rows)


levels = phishing_rate_table(df, FEATURES)
levels.to_csv(ARTIFACTS / 'feature_level_rates.csv', index=False)

# Discriminative power = how far apart a feature's own levels pull the phishing rate.
# This is a cheap univariate screen, not the feature selection itself (that is 5.1) —
# but it says up front which of the 30 columns could possibly matter.
spread = (levels.groupby('feature')['phishing_rate']
          .agg(lowest='min', highest='max')
          .eval('spread = highest - lowest')
          .sort_values('spread', ascending=False))

fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(x=spread['spread'], y=spread.index, ax=ax, color='#4C72B0')
ax.axvline(0.10, ls='--', c='crimson', lw=1)
ax.text(0.105, 0.99, '0.10 — rough "carries signal" line',
        transform=ax.get_xaxis_transform(), va='top', c='crimson', fontsize=9)
ax.set(title=f'Discriminative power of each feature\n'
             f'(spread in phishing rate across a feature\u2019s levels; overall rate {BASE_RATE:.1%})',
       xlabel='max \u2212 min phishing rate across levels', ylabel='')
plt.tight_layout()
plt.show()

spread.round(3)


In [ ]:
## Crosstabs: the six strongest features, and the three flattest for contrast
# Showing the flat ones matters as much as the strong ones — it is the evidence that
# some of the 30 columns are dead weight, which is what 5.1 acts on.

show = list(spread.index[:6]) + list(spread.index[-3:])

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=True)
for ax, f in zip(axes.ravel(), show):
    prop = pd.crosstab(df[f], df[TARGET], normalize='index')[[-1, 1]]
    prop.plot(kind='bar', stacked=True, ax=ax, legend=False,
              color=['#C44E52', '#55A868'], width=0.7)
    ax.axhline(BASE_RATE, ls='--', c='k', lw=1)
    ax.set(title=f'{f}  (spread {spread.loc[f, "spread"]:.2f})',
           xlabel='feature level', ylabel='share of sites')
    ax.tick_params(axis='x', rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ['#C44E52', '#55A868']]
fig.legend(handles, ['Phishing', 'Legitimate'], loc='lower center', ncol=2)
fig.suptitle('Strongest six features vs the three flattest \u2014 dashed line = overall phishing rate')
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

# Level meanings are defined in `Phishing Websites Features.docx`; the numeric codes
# below are not self-explanatory and must be read against it before interpreting.
levels[levels['feature'].isin(show[:6])].round(3)


> **TODO (my own words):** for each of the features that stands out, write 2-3 sentences of *security* interpretation \u2014 why an absent SSL certificate or a very young domain would correlate with phishing, and what that implies about which features the model should lean on. The rubric's A band for EDA (8 marks) is won here, not by the plots themselves. `Phishing Websites Features.docx` defines what each coded level means \u2014 use it.

What the code above found, so you know what there is to interpret:

- **`URL_of_Anchor` (spread 0.93) and `SSLfinal_State` (0.87) dominate everything else.** At their worst level 99% / 98% of sites are phishing; at their best, 6% / 11%. These two nearly separate the classes on their own \u2014 explain *why* the destination of a page's `<a>` tags and the state of its certificate would carry that much signal.
- **A middle tier of eight features** (`Prefix_Suffix`, `having_Sub_Domain`, `web_traffic`, `Links_in_tags`, `SFH`, `Request_URL`, `URL_Length`, `Domain_registeration_length`) has spread 0.24\u20130.51 \u2014 real, but not decisive alone.
- **Three features are flat: `Iframe` (0.01), `Favicon` (0.000), `popUpWidnow` (0.0001).** Their phishing rate equals the base rate at *every* level, i.e. knowing them tells you nothing. Say what you expect 5.1 to do with them \u2014 and note that flat \u2260 skewed: `Prefix_Suffix` is 87% one level and still has spread 0.51.


### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables
# Spearman rather than Pearson: these are ordered nominal codes, not measured
# quantities, so rank agreement is the only correlation that means anything here.

corr = df[FEATURES].corr(method='spearman')

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap='vlag', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.4, cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Spearman correlation between features (lower triangle)')
plt.tight_layout()
plt.show()


In [ ]:
## The pairs behind the heatmap, plus each feature's correlation with the target
pairs = (corr.where(~mask).stack().rename('corr').reset_index()
         .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'}))
pairs['abs_corr'] = pairs['corr'].abs()
top_pairs = pairs.nlargest(8, 'abs_corr')[['feature_a', 'feature_b', 'corr']]

# A correlation between two near-constant columns is driven by a handful of rows, so
# it reflects shared skew rather than shared information. Carrying each feature's
# majority-level share here stops the heatmap being read as "these are redundant"
# when the truth is "these are both almost always the same value".
skew = levels.groupby('feature')['share_of_rows'].max().rename('majority_level_share')
top_pairs = (top_pairs
             .merge(skew, left_on='feature_a', right_index=True)
             .merge(skew, left_on='feature_b', right_index=True, suffixes=('_a', '_b')))

print('Most correlated feature pairs:')
print(top_pairs.round(3).to_string(index=False))

print('\nSpearman correlation with the target (top 10 by magnitude):')
target_corr = (df[FEATURES].corrwith(df[TARGET], method='spearman')
               .sort_values(key=abs, ascending=False))
print(target_corr.head(10).round(3).to_string())


> **TODO (my own words):** call out any strongly correlated pairs and what you will do about them.

What the code above found:

- The strongest pair is **`popUpWidnow` \u2194 `Favicon` at \u03c1 = 0.94**, then `double_slash_redirecting` \u2194 `Shortining_Service` (0.84), `port` \u2194 `Favicon` (0.80), `Submitting_to_email` \u2194 `port` (0.80).
- **Every feature in that list is one of the flat, heavily skewed ones** (86\u201396% of rows sit in a single level). Two columns that are almost always `1` will look correlated whatever they measure \u2014 the coefficient is set by a few hundred rows. So the honest read is *artefact of skew*, not *redundant information*. Argue that, and say whether it changes anything about dropping one of a pair.
- **Nothing in the informative tier is strongly correlated with anything else.** `SSLfinal_State` and `URL_of_Anchor` correlate 0.74 and 0.70 with the *target* but not with each other. Say what that means for multicollinearity in LogisticRegression, and whether it changes the feature-selection plan in 5.1.


# 3. Data Preparation

## 3.1 Data Cleaning

The 30 features were extracted from raw URLs and page HTML by the original researchers and are already encoded as nominal codes. There is no raw URL string in this dataset, so there is nothing to engineer *from*.

**This project therefore uses feature selection, not feature engineering.** The spec asks for "feature engineering and/or feature selection", so selection alone is compliant — inventing transformations of already-engineered codes would be noise dressed up as work. Selection is done in section 5.1.


**Duplicates — the one real data-quality problem in this dataset.**

There are no missing values, but 11,055 rows collapse to only **5,785 unique feature vectors** (47% of rows are exact duplicates of another row). Two consequences:

1. **Leakage.** With a random train/test split, roughly half the test rows have an identical twin sitting in the training set. A tree model can memorise the twin and 'predict' it perfectly, so the reported test scores would be optimistic — they would measure recall on rows the model has effectively already seen.
2. **A hard accuracy ceiling.** 64 feature vectors appear with *both* labels (357 rows). Those are genuinely indistinguishable given these 30 features, so no model can get all of them right. That is irreducible error, not a modelling failure.

This matters more than anything else in cleaning, and it is not obvious from `df.info()` — it only shows up if you check.


In [ ]:
## Quantify the duplication before deciding what to do about it
print(f'rows                      {len(df):,}')
print(f'unique feature vectors    {df[FEATURES].drop_duplicates().shape[0]:,}')
print(f'unique rows incl. label   {df.drop_duplicates().shape[0]:,}')

conflicts = df.groupby(FEATURES)[TARGET].nunique()
print(f'feature vectors with both labels: {(conflicts > 1).sum()}')


> **TODO (my own words):** decide and defend: **drop duplicates** (honest generalisation estimate, but the dataset halves to ~5.8k rows) or **keep them** (treat repeats as real frequency information, but accept that the test score is inflated by leakage). Whichever you pick, say so explicitly and say what it costs you — a lecturer who spots the duplication will ask, and 'I didn't check' is the only wrong answer.

**Approach taken here:** build both variants, run section 4 on both, and let the measured gap decide. The evidence is below and in 4.2; what follows is what you have to argue.

Three things the numbers say, and the third is the one worth leading with:

1. **The leakage is real and large.** With duplicates kept, **65.8%** of test rows have a feature-identical twin in the training set. Deduplicated, that falls to **1.8%**. So a "test score" on the kept data is largely a score on rows the model has already seen.
2. **Deduplication flips the majority class.** Kept, the data is 55.7% legitimate; deduplicated, it is 51.6% *phishing*. So `DummyClassifier` flips from predicting legitimate (phishing recall **0.00**) to predicting phishing (phishing recall **1.00**, accuracy 0.52). Explain why a baseline with perfect recall is worthless — this is the cleanest possible argument for why you report precision, recall and F1 together rather than optimising recall alone.
3. **The inflation is model-specific, which is the interesting part.** LogisticRegression and GradientBoosting barely move (recall ±0.2pp). Only **RandomForest** is meaningfully inflated: recall 0.9612 → 0.9404, accuracy 0.9742 → 0.9419. Argue *why*: an unpruned forest can memorise individual rows, so duplicate twins reward it specifically, while a linear model and a shallow-stage booster cannot exploit them. Note the corroborating evidence — RF's train/test gap **grows** from 0.016 to 0.049 once duplicates are gone. Deduplication did not damage RF; it revealed overfitting the duplicates were hiding.

**Contradictory rows:** `drop_duplicates()` runs over features *and* label, so the 64 vectors carrying both labels survive as one row per label (128 rows, 2.2% of the deduped set). Say why that is deliberate — they are genuine ambiguity in the feature space, and deleting them would fake a ceiling the model cannot actually reach.


In [ ]:
## Clean data
# There is no missing data and no out-of-range codes, so cleaning is exactly one
# decision: what to do about the 5,206 duplicate rows.
#
# Rather than assert which choice is right, build BOTH variants and let section 4
# measure the difference. `df_dedup` is the canonical dataset carried through 5.x;
# `df_kept` exists only to quantify what keeping duplicates would have bought.

df_kept = df.copy()
df_dedup = df.drop_duplicates().reset_index(drop=True)

print(f'duplicates kept : {len(df_kept):,} rows')
print(f'deduplicated    : {len(df_dedup):,} rows  '
      f'({1 - len(df_dedup) / len(df_kept):.1%} removed)')

# drop_duplicates() is over feature+label together, so the 64 contradictory vectors
# survive as one row per label. That is deliberate: they are genuine ambiguity in the
# feature space, not dirt, and deleting them would fake a ceiling the model cannot reach.
conflicts_after = df_dedup.groupby(FEATURES)[TARGET].nunique()
print(f'contradictory vectors surviving dedup: {(conflicts_after > 1).sum()} '
      f'({(conflicts_after > 1).sum() * 2:,} rows, '
      f'{(conflicts_after > 1).sum() * 2 / len(df_dedup):.1%} of the deduped data)')

# Deduplication is not label-neutral — check before assuming the class ratio survived.
balance = pd.DataFrame({
    'duplicates kept': df_kept[TARGET].value_counts(normalize=True),
    'deduplicated': df_dedup[TARGET].value_counts(normalize=True),
}).rename(index={-1: 'phishing', 1: 'legitimate'})
print()
print(balance.round(3).to_string())


## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set
# Stratified so each test set keeps its own real class ratio — otherwise recall on the
# phishing class is measured against a distribution that does not exist.

def make_split(frame):
    return train_test_split(frame[FEATURES], frame[TARGET], test_size=0.2,
                            stratify=frame[TARGET], random_state=RANDOM_STATE)


splits = {'duplicates kept': make_split(df_kept), 'deduplicated': make_split(df_dedup)}

# The canonical split, used everywhere from 5.1 onward.
X_train, X_test, y_train, y_test = splits['deduplicated']
print(f'canonical (deduplicated): train {X_train.shape}  test {X_test.shape}')


In [ ]:
## Measure the leakage directly instead of asserting it
# The argument against keeping duplicates is that test rows reappear in training. That
# is checkable: what share of test rows have a feature-identical twin in the train set?

print('share of test rows with an identical feature vector in train:')
for name, (X_tr, X_te, _, _) in splits.items():
    train_vectors = set(map(tuple, X_tr.to_numpy()))
    overlap = np.mean([tuple(row) in train_vectors for row in X_te.to_numpy()])
    print(f'  {name:16s} {overlap:6.1%}')


# 4. Modelling

## 4.1 Baseline

`DummyClassifier(strategy='most_frequent')` always predicts the majority class. Every model below has to beat it, otherwise it has learned nothing. The rubric's A band for model comparison requires test-set results *with* a baseline comparison.


In [ ]:
## Majority-class baseline
# Fitted on both variants, because deduplication changes which class is the majority
# and therefore changes what this baseline even predicts.

for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    dummy = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
    pred = dummy.predict(X_te)
    print(f'{name:16s} predicts {dummy.classes_[dummy.predict(X_te[:1])[0] == dummy.classes_][0]:>2} '
          f'| accuracy {accuracy_score(y_te, pred):.4f} '
          f'| phishing recall {recall_score(y_te, pred, pos_label=-1, zero_division=0):.4f}')


### 4.2 Train Model

Three distinct algorithm families, all on the full 30 features (the 'before' half of the feature-selection comparison in 5.1):

- **LogisticRegression** — linear, coefficients are directly readable
- **RandomForestClassifier** — bagged trees, captures interactions
- **GradientBoostingClassifier** — boosted trees, usually strongest, slowest to train

scikit-learn only, per the spec.


In [ ]:
## Initialise and train models
# Same four estimators, both dataset variants, all 30 features. Building the models
# fresh per dataset avoids refitting an already-fitted instance.


def build_models():
    return {
        'DummyClassifier': DummyClassifier(strategy='most_frequent'),
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    }


# pos_label=-1 throughout: "positive" means *phishing*, the class whose misses are the
# expensive ones. Getting this backwards silently reports the wrong recall.
rows = []
for dataset, (X_tr, X_te, y_tr, y_te) in splits.items():
    for name, model in build_models().items():
        t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        fit_seconds = time.perf_counter() - t0
        pred_te, pred_tr = model.predict(X_te), model.predict(X_tr)
        rows.append({
            'dataset': dataset, 'model': name,
            'test_accuracy': accuracy_score(y_te, pred_te),
            'test_precision': precision_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'test_recall': recall_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'test_f1': f1_score(y_te, pred_te, pos_label=-1, zero_division=0),
            'train_accuracy': accuracy_score(y_tr, pred_tr),
            'fit_seconds': fit_seconds,
        })

baseline_metrics = pd.DataFrame(rows)
# Train minus test accuracy: the overfitting evidence for the qualitative comparison.
baseline_metrics['train_test_gap'] = (baseline_metrics['train_accuracy']
                                      - baseline_metrics['test_accuracy'])
baseline_metrics.to_csv(ARTIFACTS / 'baseline_metrics.csv', index=False)

baseline_metrics.round(4)


In [ ]:
## The duplicate decision, quantified
# Side-by-side on the metric that matters (phishing recall) plus accuracy, so the cost
# of keeping duplicates is a number rather than an assertion.

for metric in ['test_recall', 'test_accuracy', 'train_test_gap']:
    table = baseline_metrics.pivot(index='model', columns='dataset', values=metric)
    table['difference'] = table['duplicates kept'] - table['deduplicated']
    print(f'\n{metric}:')
    print(table.round(4).to_string())


> **TODO (my own words):** quantitative comparison (the metrics table) **and** qualitative: interpretability, training time, overfitting risk, what each would cost to serve in the app. Then justify which model you carry into tuning on performance *and* deployment needs — not 'highest accuracy'.

On the canonical (deduplicated) data, all 30 features:

| Model | Test recall | Test F1 | Test acc | Train−test gap | Fit time |
| --- | --- | --- | --- | --- | --- |
| DummyClassifier | 1.000 | 0.681 | 0.516 | −0.000 | 0.000s |
| LogisticRegression | 0.904 | 0.915 | 0.913 | 0.011 | 0.013s |
| RandomForest | 0.940 | 0.944 | 0.942 | **0.049** | 0.197s |
| GradientBoosting | 0.940 | 0.948 | **0.946** | 0.006 | 0.793s |

Points to argue rather than restate:

- **RF and GB tie on recall (0.9404) but are not equivalent.** GB wins slightly on F1 and accuracy *and* has an eighth of RF's train/test gap. RF is the one showing overfitting. If you pick GB, that gap — not the third decimal place of accuracy — is the reason to give.
- **GB costs 4× RF's fit time and 60× LR's.** Argue whether that matters: it is a one-off training cost, and inference is what the Streamlit app actually pays for. Do not let "slowest to train" stand in as "worst to deploy" without saying so.
- **LogisticRegression gives up ~3.6pp of recall and buys readable coefficients.** Decide honestly whether that trade is worth it here, given section 1 says a false negative puts a user on a credential-harvesting page.
- **All three beat the dummy on F1** (0.68) but only two beat it on recall, because the dummy's recall is 1.00 by construction. Use that when you justify the metric in section 5.


# 5. Model Evaluation

## Evaluation metric selection

> **TODO (my own words):** state which metric you optimise and tie it back to section 1's cost table in a few sentences. Report the full picture regardless — precision, recall, F1 and the confusion matrix — so the trade-off you accepted is visible, not asserted.


In [ ]:
## Evaluate model
## TODO: classification_report + ConfusionMatrixDisplay for the selected model.


## Iterative model development

Order matters: baselines on all 30 features (section 4) come first, so 5.1 has a genuine 'before' to compare against.


### 5.1 Feature selection

This is **feature selection, not feature engineering** — for the reason given in §3.1: the 30 columns were already extracted from raw URLs and HTML by the original researchers, and no raw URL string survives in the dataset to engineer new features from.

Two independent methods, cross-checked against each other:

1. **RandomForest feature importances** — model-based, captures interactions
2. **Mutual information** (`mutual_info_classif`, the criterion behind `SelectKBest`) — statistical, no model assumed; mutual information suits categorical features where a linear correlation would not

Agreement between two different methods is stronger evidence than either alone.

**How many features survive is decided before looking at the test set.** Both rankings and the cut are computed on the *training* split only; the size of the surviving set comes from 5-fold cross-validation *within* the training data, using the one-standard-error rule — keep the smallest feature set whose cross-validated score is within one standard error of the best one seen. Choosing `k` by re-scoring the test set would make the before/after comparison below meaningless, because the "after" would have been fitted to the very data it is reported on.


In [ ]:
## Further feature selection
# Both rankings are computed on the TRAINING split only. Ranking on the full dataset
# would let test rows influence which features exist, which is leakage by another route.
#
#   1. RandomForest impurity importances - model-based, sees interactions.
#   2. Mutual information via SelectKBest - model-free. discrete_features=True matters:
#      every column is a nominal code, so estimating MI as if -1/0/1 were points on a
#      continuous scale would be measuring the wrong thing.
#
# Neither is trusted alone. The surviving set is ordered by the two ranks averaged, and
# the two are cross-checked against each other first.

rf_ranker = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

# k='all' on purpose: SelectKBest is used here for its *scores*, not to pick the size of
# the set. Passing a k would be inventing the number this section exists to measure -
# the 1-SE cross-validation rule below decides it instead.
mi_selector = SelectKBest(
    score_func=partial(mutual_info_classif, discrete_features=True,
                       random_state=RANDOM_STATE),
    k='all').fit(X_train, y_train)

ranking = pd.DataFrame({
    'rf_importance': rf_ranker.feature_importances_,
    # Spread of the importance across the 100 trees: a large std means the forest does
    # not agree with itself about that feature, so the point estimate is not solid.
    'rf_std': np.std([t.feature_importances_ for t in rf_ranker.estimators_], axis=0),
    'mutual_info': mi_selector.scores_,
}, index=FEATURES)
ranking['rf_rank'] = ranking['rf_importance'].rank(ascending=False)
ranking['mi_rank'] = ranking['mutual_info'].rank(ascending=False)
ranking['mean_rank'] = ranking[['rf_rank', 'mi_rank']].mean(axis=1)
ranking = ranking.sort_values('mean_rank')

rho = spearmanr(ranking['rf_rank'], ranking['mi_rank']).statistic
print(f'Spearman rank correlation between the two methods: {rho:.3f}')
print('\nlargest disagreements (|rf_rank - mi_rank|):')
print((ranking['rf_rank'] - ranking['mi_rank']).abs().sort_values(ascending=False)
      .head(4).to_string())

# Section 2 predicted, from conditional phishing rates alone, that Favicon, popUpWidnow
# and Iframe carry no signal. Two model-based rankings are now in a position to falsify
# that. Printed explicitly rather than quietly hoped for.
print('\nS2 predicted three dead features. Where did the two rankings put them?')
print(ranking.loc[['Favicon', 'popUpWidnow', 'Iframe'],
                  ['rf_rank', 'mi_rank', 'mean_rank']].to_string())

ranking.to_csv(ARTIFACTS / 'feature_ranking.csv')
ranking.round(4)


In [ ]:
## Plot both rankings side by side
# Same feature order (mean rank) on both panels, so agreement and disagreement between
# the two methods is read off vertically instead of by comparing two sorted lists.

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)
order = ranking.index[::-1]  # matplotlib draws barh bottom-up

axes[0].barh(order, ranking.loc[order, 'rf_importance'],
             xerr=ranking.loc[order, 'rf_std'], color='#4C72B0',
             error_kw={'elinewidth': 0.8, 'ecolor': '#666666'})
axes[0].set_title('RandomForest importance\n(bars = mean, whiskers = std across trees)')
axes[0].set_xlabel('impurity decrease')

axes[1].barh(order, ranking.loc[order, 'mutual_info'], color='#DD8452')
axes[1].set_title('Mutual information\n(model-free)')
axes[1].set_xlabel('bits shared with the target')

fig.suptitle('Two independent feature rankings, ordered by their average rank', y=0.995)
fig.tight_layout()
plt.show()


In [ ]:
## How many features to keep - decided by cross-validation, not by taste
# Walk k = 1..30 down the averaged ranking and cross-validate inside the training split.
# All three models are swept, not just the one carried forward: if the curves plateau in
# the same place, the cut is a property of the data, not of one estimator's quirks.
#
# Takes ~100s - it is 30 x 3 x 5 = 450 fits, and GradientBoosting is the slow one.

ordered_features = list(ranking.index)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'recall': make_scorer(recall_score, pos_label=-1, zero_division=0),
           'f1': make_scorer(f1_score, pos_label=-1, zero_division=0)}

rows = []
for k in range(1, len(ordered_features) + 1):
    cols = ordered_features[:k]
    for name, model in build_models().items():
        if name == 'DummyClassifier':
            continue
        cvres = cross_validate(model, X_train[cols], y_train, cv=cv, scoring=scoring)
        rows.append({'k': k, 'model': name,
                     'cv_recall': cvres['test_recall'].mean(),
                     'cv_recall_se': cvres['test_recall'].std(ddof=1) / np.sqrt(cv.get_n_splits()),
                     'cv_f1': cvres['test_f1'].mean(),
                     'cv_f1_se': cvres['test_f1'].std(ddof=1) / np.sqrt(cv.get_n_splits())})

sweep = pd.DataFrame(rows)
sweep.to_csv(ARTIFACTS / 'feature_sweep.csv', index=False)


def smallest_k_within_1se(frame, mean_col, se_col):
    """One-standard-error rule: of every k whose mean score reaches (best - 1 SE of the
    best), take the smallest. Prefers the simpler model when the difference is inside
    the noise of the cross-validation itself."""
    best = frame.loc[frame[mean_col].idxmax()]
    return int(frame.loc[frame[mean_col] >= best[mean_col] - best[se_col], 'k'].min()), best


print('smallest k within 1 SE of that model\'s own best CV score:')
for name in sweep['model'].unique():
    s = sweep[sweep.model == name]
    k_f1, b_f1 = smallest_k_within_1se(s, 'cv_f1', 'cv_f1_se')
    k_re, b_re = smallest_k_within_1se(s, 'cv_recall', 'cv_recall_se')
    print(f'  {name:20s} by F1: k={k_f1:2d}  (best {b_f1.cv_f1:.4f} at k={int(b_f1.k)})'
          f'   | by recall: k={k_re:2d}  (best {b_re.cv_recall:.4f} at k={int(b_re.k)})')

# The model carried forward from section 4.2. One constant, deliberately: if the
# argument in 4.2 lands on a different model, change it here and 5.1-5.3 follow.
SELECTED_MODEL = 'GradientBoosting'

# Cut on F1, not on recall alone. Recall by itself would pick a much smaller set (see
# above) because recall can be bought by predicting 'phishing' more often - the dummy in
# section 4.1 scores 1.000 on it. F1 keeps precision in the objective, so the surviving
# set has to earn its size on both classes.
selected_curve = sweep[sweep.model == SELECTED_MODEL]
K, best = smallest_k_within_1se(selected_curve, 'cv_f1', 'cv_f1_se')
SELECTED_FEATURES = ordered_features[:K]
DROPPED_FEATURES = ordered_features[K:]

print(f'\nselected model : {SELECTED_MODEL}')
print(f'best CV F1     : {best.cv_f1:.4f} at k={int(best.k)}, SE {best.cv_f1_se:.4f}'
      f'  -> keep any k scoring >= {best.cv_f1 - best.cv_f1_se:.4f}')
print(f'chosen k       : {K}   ({len(DROPPED_FEATURES)} features dropped)')
print(f'\nkept    : {SELECTED_FEATURES}')
print(f'\ndropped : {DROPPED_FEATURES}')

json.dump({'selected_model': SELECTED_MODEL, 'k': K,
           'selection_rule': 'smallest k within 1 SE of best 5-fold CV F1 on the '
                             'training split, features ordered by mean(RF importance '
                             'rank, mutual information rank)',
           'features': SELECTED_FEATURES, 'dropped': DROPPED_FEATURES},
          open(ARTIFACTS / 'selected_features.json', 'w'), indent=2)


In [ ]:
## The two curves that justify the cut
# Left: every model's F1 against k, so the plateau can be seen to be shared.
# Right: recall vs F1 for the selected model - they saturate at very different k, which
# is the whole reason the cut is made on F1.

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name in sweep['model'].unique():
    s = sweep[sweep.model == name]
    axes[0].plot(s['k'], s['cv_f1'], marker='o', ms=3, label=name)
axes[0].axhline(best.cv_f1 - best.cv_f1_se, ls=':', c='grey',
                label=f'{SELECTED_MODEL} best - 1 SE')
axes[0].axvline(K, ls='--', c='crimson', label=f'chosen k = {K}')
axes[0].set(xlabel='number of features kept (k)', ylabel='5-fold CV F1 (phishing)',
            title='All three models plateau in the same region')
axes[0].legend(fontsize=8)

s = selected_curve
axes[1].plot(s['k'], s['cv_recall'], marker='o', ms=3, label='recall', c='#C44E52')
axes[1].plot(s['k'], s['cv_f1'], marker='o', ms=3, label='F1', c='#4C72B0')
axes[1].axvline(K, ls='--', c='crimson')
axes[1].set(xlabel='number of features kept (k)', ylabel='5-fold CV score',
            title=f'{SELECTED_MODEL}: recall saturates long before F1 does')
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()


In [ ]:
## Before / after: all 30 features vs reduced set
# Refit on the reduced set and score the test set once. The 'before' half is read back
# from section 4's saved metrics rather than recomputed, so the two halves are
# guaranteed to come from the same split and the same fitted-once protocol.

before = pd.read_csv(ARTIFACTS / 'baseline_metrics.csv')
before = before[(before.dataset == 'deduplicated') & (before.model != 'DummyClassifier')]

rows = []
for name, model in build_models().items():
    if name == 'DummyClassifier':
        continue
    t0 = time.perf_counter()
    model.fit(X_train[SELECTED_FEATURES], y_train)
    fit_seconds = time.perf_counter() - t0
    pred_te = model.predict(X_test[SELECTED_FEATURES])
    pred_tr = model.predict(X_train[SELECTED_FEATURES])
    rows.append({
        'model': name,
        'test_accuracy': accuracy_score(y_test, pred_te),
        'test_precision': precision_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'test_recall': recall_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'test_f1': f1_score(y_test, pred_te, pos_label=-1, zero_division=0),
        'train_accuracy': accuracy_score(y_train, pred_tr),
        'fit_seconds': fit_seconds,
    })

reduced_metrics = pd.DataFrame(rows)
reduced_metrics['train_test_gap'] = (reduced_metrics['train_accuracy']
                                     - reduced_metrics['test_accuracy'])
reduced_metrics['n_features'] = K
reduced_metrics.to_csv(ARTIFACTS / 'reduced_metrics.csv', index=False)

COMPARE_ON = ['test_recall', 'test_precision', 'test_f1', 'test_accuracy', 'train_test_gap']
comparison = (before.set_index('model')[COMPARE_ON]
              .join(reduced_metrics.set_index('model')[COMPARE_ON],
                    lsuffix=' (30)', rsuffix=f' ({K})'))

print(f'canonical deduplicated test set - 30 features vs {K}:')
for metric in COMPARE_ON:
    pair = comparison[[f'{metric} (30)', f'{metric} ({K})']].copy()
    pair['change'] = pair[f'{metric} ({K})'] - pair[f'{metric} (30)']
    print(f'\n{metric}:')
    print(pair.round(4).to_string())


> **TODO (my own words):** did performance hold, improve, or trade off? Tie the decision to reduced complexity and to how many inputs the Streamlit user has to fill in — not to a metric number alone. This row is worth 10 marks, the largest single one.

**What was measured.** Averaged ranking of RF importance and mutual information; the two agree strongly (Spearman ρ = **0.899**). The one-SE rule on 5-fold CV F1 inside the training split cut at **k = 16**, dropping 14 features.

Test set, GradientBoosting, 30 features → 16:

| Metric | 30 features | 16 features | Change |
| --- | --- | --- | --- |
| Recall (phishing) | 0.9404 | 0.9387 | −0.0017 |
| Precision | 0.9546 | 0.9562 | +0.0015 |
| F1 | 0.9475 | 0.9474 | −0.0001 |
| Accuracy | 0.9462 | 0.9462 | 0.0000 |
| Train−test gap | 0.0062 | **0.0013** | −0.0049 |

Points to argue rather than restate:

- **Performance held; it did not improve.** F1 moved by one ten-thousandth and accuracy not at all. The honest claim is "same accuracy on half the inputs", not "feature selection made the model better". The one thing that did move is the train−test gap, down 5× — a smaller hypothesis space generalises a little more tightly.
- **The 14 dropped features cost the user 14 dropdowns.** That is the real return here, and §7 of the guideline explicitly wants the app's inputs to be the selected features. Argue the deployment case, not the metric case.
- **The two rankings agree, and they agree with §2.** `Favicon` (24.5), `popUpWidnow` (24.0) and `Iframe` (29.0) all landed in the bottom seven by mean rank — the S2 prediction from conditional phishing rates alone survived contact with two model-based methods. Worth saying out loud at the demo: the prediction was recorded *before* this was run.
- **Where the methods disagreed is more interesting than where they agreed.** `Submitting_to_email` (RF 21st, MI 30th), `Shortining_Service` and `Abnormal_URL` (RF 24th/23rd, MI 15th/14th) split by 9 ranks. MI is univariate; RF sees a feature used inside a split that another feature has already conditioned on. All three fall below the cut anyway, so nothing rides on it — say so, rather than over-claiming.
- **Cutting on F1 rather than recall was a choice with a consequence.** By recall alone the one-SE rule would have kept **k = 5** for GradientBoosting — recall is already 0.9354 at five features and only reaches 0.9395 at thirty. Features 6–16 are buying *precision*, not recall. Defend why that is worth 11 extra dropdowns: recall alone is gameable (the dummy in §4.1 scores 1.000 on it), so a set chosen to maximise recall would be chosen to maximise a number that a useless model already wins.
- **The cut is not knife-edge, and pretending otherwise would be false precision.** k = 13 scores 0.9409 against a threshold of 0.9410. The curve is flat from roughly k = 13 to k = 30; the rule picked 16, but any k in the teens is defensible. Better to state that than to imply 16 is optimal.
- **Watch the model-specific caveat.** The rule gives k = 8 for LogisticRegression and k = 21 for RandomForest. The cut is tuned to the model §4.2 carries forward. If that argument lands on a different model, `SELECTED_MODEL` in the cell above has to change and this table has to be re-run.


### 5.2 Hyperparameter tuning

`RandomizedSearchCV` only, **exactly 2 hyperparameters, at most 3 values each** — a hard spec constraint. `random_state=RANDOM_STATE` throughout for reproducibility.


In [ ]:
## RandomizedSearchCV
## TODO: log the grid, the full cv_results_ table -> artifacts/search_results.csv,
## and the best params.


In [ ]:
## Tuned vs untuned on the test set
## TODO: side-by-side table, not a sentence. The rubric asks for documented
## improvement vs baseline.


> **TODO (my own words):** explain the impact and why these final settings.


### 5.3 Final model


In [ ]:
## Persist for the Streamlit app
## TODO: joblib.dump the fitted model -> artifacts/final_model.pkl
## and the feature list the app must collect, in the same column order.


In [ ]:
## New data

## Predict
## TODO: sanity-check one hand-built phishing-looking row and one legitimate-looking
## row. This is also the smoke test for the app in S8.
